**Lab type:** review  
**Course:** NL301 Natural Language Processing with Python  
**Lesson:** 03 — Text Representations  
**Task:** Review a TF-IDF classification pipeline and answer five questions about representation choices, leakage, and failure modes.

## Setup

In [ ]:
!pip install scikit-learn numpy --quiet
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Simulated imbalanced movie-review dataset (185 positive, 15 negative)
np.random.seed(42)
pos = [f"great movie loved every moment scene {i}" for i in range(185)]
neg = [f"terrible film boring waste of time {i}" for i in range(15)]
texts  = pos + neg
labels = [1]*185 + [0]*15


## The pipeline to review

In [ ]:
# Pipeline under review — read carefully before answering the questions below.
vectorizer = TfidfVectorizer(max_features=500)
X = vectorizer.fit_transform(texts)                  # (A) fit on full dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42)

clf = LogisticRegression()
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))


---
## Review Question 1: Data leakage via early vectorisation

`vectorizer.fit_transform(texts)` is called on the full dataset **before** `train_test_split`. What leakage does this introduce? Write the corrected code using a `Pipeline`.

<details>
<summary>🔑 Reveal answer — Q1</summary>

**The leakage:** `vectorizer.fit_transform(texts)` is called on the entire dataset before `train_test_split`. This means the fitted vocabulary and IDF weights incorporate information from test documents — their unique tokens and their document-frequency counts. At evaluation time, the model has implicitly "seen" test data during preprocessing, producing an optimistic score that won't generalise.

**Correct approach:** Use a sklearn `Pipeline([('tv', TfidfVectorizer(...)), ('clf', ...)])` and call `pipe.fit(X_train, y_train)`. The pipeline's `fit_transform` only ever runs on training data; test data goes through `transform` using the training-fitted vocabulary. With `cross_val_score`, this is enforced across all folds automatically.

</details>

In [ ]:
# Your analysis and corrected code here
from sklearn.pipeline import Pipeline

# Demonstrate the leakage effect, then fix it


---
## Review Question 2: CountVectorizer vs TF-IDF

Switch the pipeline to use `CountVectorizer` instead of `TfidfVectorizer`. What does IDF weighting add? In which scenarios does raw count representation outperform TF-IDF?

<details>
<summary>🔑 Reveal answer — Q2</summary>

**What IDF adds:** TF-IDF divides term frequency by document frequency (log-scaled), down-weighting tokens that appear in nearly every document (e.g., "movie", "film", "the"). These common tokens would otherwise dominate cosine similarity despite carrying no discriminative signal. IDF amplifies rare, discriminative terms.

**When CountVectorizer can win:** On very short documents where IDF normalisation degrades already-sparse representations; in spam detection where high raw frequency of identical phrases (e.g., "click here", "free prize") is itself the signal; or when the corpus is homogeneous enough that document frequency is uniformly high, making IDF weights flat and uninformative.

</details>

In [ ]:
# Compare CountVectorizer vs TfidfVectorizer
from sklearn.pipeline import Pipeline

pipe_count = Pipeline([('cv', CountVectorizer(max_features=500)), ('clf', LogisticRegression())])
pipe_tfidf = Pipeline([('tv', TfidfVectorizer(max_features=500)), ('clf', LogisticRegression())])

X_tr, X_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.2, random_state=42)
for name, pipe in [('Count', pipe_count), ('TF-IDF', pipe_tfidf)]:
    pipe.fit(X_tr, y_tr)
    print(f"{name} accuracy: {pipe.score(X_te, y_te):.3f}")


---
## Review Question 3: Misleading accuracy on imbalanced data

The pipeline above reports 0.94 accuracy on a 185 positive / 15 negative dataset. Is this trustworthy? Calculate the majority-class baseline. What metric should you use instead?

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Why accuracy is misleading:** The majority-class baseline is 185/200 = 0.925 — a classifier that always predicts "positive" achieves 92.5% accuracy without learning anything. If the pipeline reports 0.94, the improvement over a trivial baseline is only 1.5 points. Accuracy rewards predicting the dominant class and completely hides failures on the minority class.

**Better metric:** Macro F1 weights each class equally regardless of support, so a fully missed negative class (F1=0.0) drags the macro average down significantly. Minority-class F1 (the negative class F1 alone) is the most direct measure of what you actually care about.

</details>

In [ ]:
# Calculate majority-class baseline accuracy
total = len(labels)
majority = max(sum(labels), total - sum(labels))
print(f"Majority-class baseline: {majority/total:.3f}")

# Then evaluate with a better metric
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline

pipe = Pipeline([('tv', TfidfVectorizer(max_features=500)), ('clf', LogisticRegression())])
X_tr, X_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.2, random_state=42)
pipe.fit(X_tr, y_tr)
preds = pipe.predict(X_te)
print("F1 (macro):", f1_score(y_te, preds, average='macro'))
print("F1 (weighted):", f1_score(y_te, preds, average='weighted'))


---
## Review Question 4: Out-of-vocabulary words

A new review contains the word `"unforgettable"`, which was not in the training vocabulary. What happens to its TF-IDF representation? Demonstrate and explain.

<details>
<summary>🔑 Reveal answer — Q4</summary>

**What happens:** `TfidfVectorizer` silently ignores any token not in its fitted vocabulary. "Unforgettable" was not in the training corpus, so its TF-IDF weight is 0 across all dimensions — the word contributes nothing to the document's vector. The review is classified using only the remaining tokens that were seen at training time.

**Implication:** Rare or domain-specific vocabulary is invisible to bag-of-words models. If the most important word in a document is OOV, the classifier may produce a confident but wrong prediction. Sentence embeddings avoid this because subword tokenisation handles unseen compound words.

</details>

In [ ]:
# Fit vectorizer on training set only
pipe = Pipeline([('tv', TfidfVectorizer(max_features=500)), ('clf', LogisticRegression())])
pipe.fit(X_tr, y_tr)

new_review = "the ending was completely unforgettable and deeply moving"
vec = pipe.named_steps['tv']
transformed = vec.transform([new_review])
print("Non-zero features:", transformed.nnz)
print("'unforgettable' in vocab:", 'unforgettable' in vec.vocabulary_)
# What does this mean for the classification?


---
## Review Question 5: Semantic similarity failure

You want to find reviews semantically similar to `"terrible shipping experience"`. TF-IDF retrieves nothing useful because `"delivery was a disaster"` shares no tokens. Why does TF-IDF fail here? What representation would you use instead?

<details>
<summary>🔑 Reveal answer — Q5</summary>

**Why TF-IDF fails here:** TF-IDF represents text as a sparse vector over a fixed vocabulary. "Terrible shipping experience" and "delivery was a disaster completely unacceptable" share zero token overlap, so their cosine similarity is 0.0. TF-IDF is a lexical method — it cannot match synonyms, paraphrases, or semantically related phrases that use different words.

**What to use instead:** Sentence embeddings (e.g., `all-MiniLM-L6-v2`) encode meaning into dense vectors trained on large corpora. Semantically related phrases like "terrible shipping" and "delivery disaster" are mapped to nearby vectors because the model has learned that these phrases occur in similar contexts, placing them close in embedding space regardless of surface vocabulary.

</details>

In [ ]:
# Demonstrate TF-IDF's lexical matching limitation
from sklearn.metrics.pairwise import cosine_similarity

query = "terrible shipping experience"
corpus = [
    "delivery was a disaster completely unacceptable",
    "terrible shipping experience very slow",
    "great movie loved every moment",
]
tv = TfidfVectorizer()
X_corpus = tv.fit_transform(corpus)
q_vec = tv.transform([query])
sims = cosine_similarity(q_vec, X_corpus)[0]
for doc, sim in zip(corpus, sims):
    print(f"  {sim:.3f}  {doc}")
print()
print("'delivery' in vocab:", 'delivery' in tv.vocabulary_)
print("'terrible' in vocab:", 'terrible' in tv.vocabulary_)
# Why do semantically similar docs score 0? What would sentence embeddings do differently?
